# Module 20: Spatial Transcriptomics
**Tier 3 — Applied Bioinformatics | Module 20**
Prerequisites: Module 03 (RNA-seq), Module 07 (ML/Clustering), Module 12 (Single-cell Scanpy).

---

**By the end you will be able to:**
- Load spatial AnnData with coordinates
- Perform QC and normalization for spatial data
- Build spatial neighbor graphs
- Detect spatially variable genes (Moran's I)
- Visualize expression overlaid on tissue space

**Attribution:** *Patterns inspired by NGSchool 2023 spatial transcriptomics practical. Uses public 10x Visium mouse brain dataset via Squidpy.*

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: GWAS](../19_GWAS/19_gwas.ipynb) | [Next: Copy Number Analysis →](../21_Copy_Number_Analysis/21_copy_number_analysis.ipynb)

---

## Background: Spatial Transcriptomics

Unlike bulk or single-cell RNA-seq, spatial methods preserve the physical location of each measurement on a tissue section.

**Key platforms:**

| Platform | Resolution | Throughput | Type |
|---|---|---|---|
| 10x Visium | 55 µm spot (~10–20 cells) | ~5,000 spots | Capture-based |
| Xenium | Single-cell (~10 µm) | ~200,000 cells | In situ sequencing |
| Slide-seq | ~10 µm | ~50,000 beads | Capture-based |

**AnnData for spatial data:**
- `adata.obs` — spot/cell metadata (tissue_section, in_tissue flag)
- `adata.obsm["spatial"]` — (n_spots, 2) pixel coordinates
- `adata.uns["spatial"]` — tissue image and scale factor metadata
- `adata.X` — count matrix (n_spots × n_genes)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    import squidpy as sq
    import scanpy as sc
    print(f"squidpy {sq.__version__}, scanpy {sc.__version__}")
except ImportError:
    print("Install: pip install squidpy scanpy")

plt.rcParams.update({"figure.dpi":110})
sc.settings.verbosity = 0

In [ ]:
# Load public mouse brain H&E Visium dataset (shipped with Squidpy)
adata = sq.datasets.visium_hne_adata()
print(adata)
print(f"\nSpot coordinates shape: {adata.obsm['spatial'].shape}")
print(f"Genes: {adata.n_vars}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")

# Show the tissue image
fig, ax = plt.subplots(figsize=(5,5))
sq.pl.spatial_scatter(adata, ax=ax, size=0.8, title="Raw spots on tissue")
plt.tight_layout(); plt.show()

In [ ]:
# QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(14,4))
axes[0].hist(adata.obs["total_counts"], bins=40, color="steelblue")
axes[0].set_xlabel("Total counts"); axes[0].set_title("Counts per spot")
axes[1].hist(adata.obs["n_genes_by_counts"], bins=40, color="steelblue")
axes[1].set_xlabel("Genes detected"); axes[1].set_title("Genes per spot")
axes[2].hist(adata.obs["pct_counts_mt"], bins=40, color="salmon")
axes[2].set_xlabel("% MT counts"); axes[2].set_title("Mitochondrial fraction")
plt.tight_layout(); plt.show()

# Filter
sc.pp.filter_cells(adata, min_counts=200)
sc.pp.filter_genes(adata, min_cells=5)
print(f"After QC: {adata.n_obs} spots, {adata.n_vars} genes")

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)
print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

sc.pp.pca(adata, n_comps=50, use_highly_variable=True)
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.leiden(adata, resolution=0.5, key_added="leiden_0.5")
sc.tl.umap(adata)

fig, axes = plt.subplots(1, 2, figsize=(12,5))
sc.pl.umap(adata, color="leiden_0.5", ax=axes[0], show=False, title="UMAP clusters")
sq.pl.spatial_scatter(adata, color="leiden_0.5", ax=axes[1], size=1.4, title="Spatial clusters")
plt.tight_layout(); plt.show()

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="visium", n_rings=1)
print("Spatial graph built.")
print(f"Connectivity matrix shape: {adata.obsp['spatial_connectivities'].shape}")
print(f"Mean neighbors per spot: {adata.obsp['spatial_connectivities'].sum(1).mean():.1f}")

In [ ]:
sq.gr.spatial_autocorr(adata, mode="moran", genes=adata.var_names[:500], n_perms=100)
moran = adata.uns["moranI"].sort_values("I", ascending=False)
print("Top 10 spatially variable genes (Moran's I):")
print(moran.head(10))

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(range(len(moran)), moran["I"], s=5, alpha=0.5, color="steelblue")
ax.set_xlabel("Gene rank"); ax.set_ylabel("Moran's I")
ax.set_title("Spatial autocorrelation (Moran's I) — all genes")
plt.tight_layout(); plt.show()

In [ ]:
top_svgs = moran.head(6).index.tolist()
print("Plotting top SVGs:", top_svgs)
fig = sq.pl.spatial_scatter(adata, color=top_svgs, ncols=3, size=1.4,
                            img_alpha=0.4, return_fig=True)
if fig: fig.suptitle("Top spatially variable genes", y=1.01)
plt.tight_layout(); plt.show()

## Cell-type Deconvolution

Each Visium spot captures RNA from ~10–20 cells. To infer cell-type composition of each spot, **deconvolution** methods use a scRNA-seq reference.

**Key tools:**

| Tool | Approach | Reference needed |
|---|---|---|
| RCTD (robust cell type decomposition) | Poisson GLM | Yes (scRNA-seq) |
| cell2location | Negative binomial + hierarchical | Yes |
| Stereoscope | Probabilistic | Yes |
| NNLS | Non-negative least squares | Yes (marker genes) |

**Conceptual pattern:**
```python
# RCTD (spacexr package — R, or Python wrapper)
# 1. Build reference: scRNA-seq adata with cell_type labels
# 2. Run deconvolution on spatial adata
# 3. adata.obsm["cell_type_proportions"] shape: (n_spots, n_types)
# 4. sq.pl.spatial_scatter(adata, color=adata.obsm["cell_type_proportions"].columns)
```

## Summary

| Step | Tool | Key Output |
|---|---|---|
| Load Visium | `sq.datasets.visium_hne_adata()` | AnnData with spatial coords |
| QC | `sc.pp.filter_cells/genes` | Clean count matrix |
| Normalize | `sc.pp.normalize_total + log1p` | Log-normalized expression |
| Cluster | `sc.tl.leiden` | Spot clusters |
| Spatial graph | `sq.gr.spatial_neighbors` | `obsp["spatial_connectivities"]` |
| SVGs | `sq.gr.spatial_autocorr(mode="moran")` | Moran's I per gene |
| Visualize | `sq.pl.spatial_scatter` | Gene expression on tissue |

**Related skill:** `spatial-transcriptomics.md`

In [ ]:
# Exercise 20
# 1. Identify the top 3 marker genes for each leiden cluster (sc.tl.rank_genes_groups).
#    Visualize them with sq.pl.spatial_scatter. Do clusters correspond to tissue regions?
# 2. Change the leiden resolution from 0.5 to 1.0 and 0.3. How does the number of
#    clusters change? Which resolution best matches the tissue anatomy?
# 3. Compute Moran's I for the top 5 marker genes from exercise 1.
#    Are highly spatially variable genes the same as cluster markers?
# 4. Challenge: implement a simple NNLS deconvolution using two "pure" reference
#    gene signatures (you can define them manually). Visualize the deconvolution
#    result spatially.

---

[← Previous: GWAS](../19_GWAS/19_gwas.ipynb) | [Next: Copy Number Analysis →](../21_Copy_Number_Analysis/21_copy_number_analysis.ipynb)

---